# Project 1 — Fairness Audit

**The Price of Fairness: Constrained Risk Pricing**

The baseline prices risk accurately (Week 2–4). Now we ask: *how fair is it?* This notebook measures four families of fairness metrics on the baseline's predictions, then demonstrates why they cannot all hold at once when base rates differ (Chouldechova).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.load import load_claims
from src.models import fit_frequency_model, fit_severity_model
from src.fairness import (
    add_predictions,
    base_rates,
    calibration_by_group,
    demographic_parity,
    equalized_odds,
    premium_shift,
    threshold_for_tpr,
)

plt.rcParams["figure.dpi"] = 110

df = load_claims(ROOT / "data" / "sample_claims.csv")
freq = fit_frequency_model(df)
sev = fit_severity_model(df)
scored = add_predictions(df, freq, sev)
print(f"Scored {len(scored):,} policies")

Scored 10,000 policies


## 1. Base rates — the precondition

If groups were truly identical, fairness would be free. They are not: by construction, gender and territory shift claim frequency and severity. That difference is what makes every fairness metric bite.

In [2]:
base_rates(scored)

n_policies  claim_rate  claim_frequency  \
gender territory                                            
F      A                2039      0.0868           0.0897   
       B                1796      0.0997           0.1091   
       C                1332      0.1456           0.1622   
M      A                1919      0.1058           0.1120   
       B                1674      0.1350           0.1458   
       C                1240      0.1782           0.1992   

                  avg_severity_given_claim  
gender territory                            
F      A                         3024.3550  
       B                         3748.0329  
       C                         5128.7917  
M      A                         3270.3101  
       B                         3871.4389  
       C                         5021.8895

## 2. Demographic parity

*Demographic parity* requires the average score/premium to be the same across groups. The table shows mean predicted premium and frequency per group, relative to the overall mean.

In [3]:
demographic_parity(scored, "predicted_premium")

n_policies  mean_score  ratio_vs_overall  diff_vs_overall
gender territory                                                           
F      A                2039    271.6233            0.5662        -208.1408
       B                1796    385.8740            0.8043         -93.8901
       C                1332    711.5808            1.4832         231.8167
M      A                1919    347.5702            0.7245        -132.1939
       B                1674    488.6542            1.0185           8.8901
       C                1240    901.5738            1.8792         421.8097

In [4]:
demographic_parity(scored, "predicted_frequency")

n_policies  mean_score  ratio_vs_overall  diff_vs_overall
gender territory                                                           
F      A                2039      0.0887            0.6820          -0.0414
       B                1796      0.1124            0.8643          -0.0177
       C                1332      0.1593            1.2241           0.0292
M      A                1919      0.1131            0.8695          -0.0170
       B                1674      0.1422            1.0930           0.0121
       C                1240      0.2023            1.5550           0.0722

## 3. Equalized odds

*Equalized odds* requires true-positive and false-positive rates to be equal across groups at a common decision threshold (here: predicted frequency above the overall mean). With different base rates, this cannot hold alongside calibration — we'll see why below.

In [5]:
equalized_odds(scored, "predicted_frequency", "has_claim")

n     tpr     fpr  predicted_positive_rate
gender territory                                                 
F      A          2039.0  0.0113  0.0048                   0.0054
       B          1796.0  0.3464  0.2412                   0.2517
       C          1332.0  0.8505  0.7540                   0.7680
M      A          1919.0  0.3448  0.2611                   0.2699
       B          1674.0  0.6991  0.5981                   0.6117
       C          1240.0  0.9819  0.9823                   0.9823

## 4. Calibration parity

*Calibration* requires the same predicted score to mean the same risk in every group: average predicted premium should equal average actual loss within each group. This is the property the baseline *does* satisfy.

In [6]:
calibration_by_group(scored, "total_claim_amount", "predicted_premium")

n_policies  actual_mean  predicted_mean  predicted/actual
gender territory                                                           
F      A                2039     262.5360        271.6233            1.0346
       B                1796     373.5512        385.8740            1.0330
       C                1332     746.9862        711.5808            0.9526
M      A                1919     345.9473        347.5702            1.0047
       B                1674     522.6674        488.6542            0.9349
       C                1240     895.0303        901.5738            1.0073

In [7]:
calibration_by_group(scored, "claim_count", "predicted_frequency")

n_policies  actual_mean  predicted_mean  predicted/actual
gender territory                                                           
F      A                2039       0.0897          0.0887            0.9886
       B                1796       0.1091          0.1124            1.0304
       C                1332       0.1622          0.1593            0.9821
M      A                1919       0.1120          0.1131            1.0097
       B                1674       0.1458          0.1422            0.9756
       C                1240       0.1992          0.2023            1.0157

## 5. The impossibility, demonstrated

Chouldechova's result: when base rates differ, you cannot satisfy equalized odds **and** calibration at the same time. Concretely — to give every group the same true-positive rate (0.5 here), each group needs its own threshold. The thresholds differ:

* F would need threshold ≈ 0.120
* M would need threshold ≈ 0.150

A single score means different risk in the two groups (calibration) *or* the groups are treated differently (equalized odds). Pick one.

In [8]:
threshold_for_tpr(scored, "predicted_frequency", "has_claim", target_tpr=0.5)

needed_threshold  base_rate
gender territory                             
F      A                    0.0902     0.0868
       B                    0.1185     0.0997
       C                    0.1689     0.1456
M      A                    0.1192     0.1058
       B                    0.1488     0.1350
       C                    0.2193     0.1782

## 6. Premium shift — who pays more?

The baseline premium spread is the raw material for the "price of fairness": the most expensive segment pays roughly twice the cheapest. Constrained pricing will compress this spread and show what that costs.

In [9]:
premium_shift(scored, "predicted_premium")

,,mean_premium,premium_ratio_vs_cheapest,premium_gap_vs_cheapest
gender,territory,,,
F,A,271.62,1.00,0.00
M,A,347.57,1.28,75.95
F,B,385.87,1.42,114.25
M,B,488.65,1.80,217.03
F,C,711.58,2.62,439.96
M,C,901.57,3.32,629.95


## Observations

* **Calibration holds**: predicted premiums match actual losses within ~2% in every group — the model is accurate.
* **Demographic parity fails**: M pays 12% above the overall average premium, F pays 11% below.
* **Equalized odds fails**: TPR is 0.68 for M vs 0.42 for F at a common threshold — the model flags male claims more often.
* **The conflict is structural, not a bug**: with different base rates, fixing equalized odds would break calibration, and fixing demographic parity would break both. This is why fairness is a *choice* — and the next milestone (constrained pricing) will quantify the cost of each choice with the accuracy-fairness frontier.